# Random-Forest Baseline v2

Dieses Notebook zeigt die optimierte zweite Version der Random-Forest-Baseline inklusive Search-Ergebnissen, Modellmetriken und grafischer Confusion Matrix.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src" / "random_forest_baseline_v2").exists():
            return candidate
    raise FileNotFoundError("Repo root mit src/random_forest_baseline_v2 wurde nicht gefunden.")

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.random_forest_baseline.config import BaselineConfig
from src.random_forest_baseline.data import prepare_dataset
from src.random_forest_baseline.modeling import evaluate_grouped_dataset, save_confusion_matrix_plot

config = BaselineConfig.from_json(repo_root / "baseline_models/random_forest_v2/config.json")
config

In [ ]:
search_results_path = repo_root / "baseline_models/random_forest_v2/output/search_results.csv"
search_summary_path = repo_root / "baseline_models/random_forest_v2/output/search_summary.json"
if search_results_path.exists():
    display(pd.read_csv(search_results_path).head(10))
if search_summary_path.exists():
    display(pd.DataFrame([json.loads(search_summary_path.read_text())]))

In [ ]:
dataset = prepare_dataset(config)
display(dataset.label_summary)
display(dataset.metadata.head())
print(f"Events: {len(dataset.labels)} | Sessions: {dataset.groups.nunique()} | Features: {dataset.features.shape[1]}")

In [ ]:
result = evaluate_grouped_dataset(dataset, config)
display(pd.DataFrame([result.overall_metrics]))
display(result.classification_report)
display(result.fold_metrics)

In [ ]:
figure = save_confusion_matrix_plot(result.confusion_matrix)
figure